In [1]:
# CASSANDRA setup cell — run this FIRST, in order, before any other cell.
# On Colab with GPU desired (Pythia extraction): Runtime > Change runtime
# type > GPU, THEN run this cell.
import importlib
import os
import subprocess
import sys
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules
REPO_URL = "{repo_url}"

if IN_COLAB:
    if not Path("/content/CASSANDRA/.git").exists():
        subprocess.run(["git", "clone", REPO_URL, "/content/CASSANDRA"],
                       check=True)
    os.chdir("/content/CASSANDRA")
    subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                    "-r", "requirements.txt"], check=True)
    subprocess.run([sys.executable, "-m", "spacy", "download",
                    "en_core_web_sm"], check=True, capture_output=True)
    subprocess.run([sys.executable, "scripts/prepare_data.py",
                    "--config", "config.yaml"], check=True)
    print("Colab setup complete. Working directory:", os.getcwd())
else:
    try:
        import torch, transformer_lens, sklearn, pandas, seaborn  # noqa: F401
        print("Local environment OK. Run `make data` if data/processed "
              "is empty.")
    except ImportError as exc:
        raise SystemExit(
            f"Missing dependency ({{exc}}). Run `make setup` from the "
            "repository root before executing this notebook."
        )


/home/z/my-project/cassandra/.venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Local environment OK. Run `make data` if data/processed is empty.


# 04 — Generation case study

**Objective.** The qualitative face of the quantitative claims: apply the best probe of notebook 03 to free generations over TriviaQA, compare internal confidence against real factual agreement, and build the selective-abstention policy.

**Data.** Greedy (temperature 0.0) generations of max_new_tokens tokens for n_prompts TriviaQA questions per model, scored by the best probe configuration; ground truth by normalized string-match against the dataset aliases, with ambiguous matches excluded and counted.

**Reading guide.** Confusion-internal-vs-external is the point: a high probe score on an incorrect generation is a captured confabulation candidate; a low score on a correct one is atypical doubt.

## Environment and determinism

In [2]:
import numpy as np
import pandas as pd

SEED = 42
np.random.seed(SEED)
rng = np.random.default_rng(SEED)

pd.set_option("display.max_colwidth", 120)


In [3]:
import sys

import matplotlib
import matplotlib.pyplot as plt
import seaborn as sns

from cassandra_ns.report.figures import save_figure

if "google.colab" not in sys.modules:
    matplotlib.use("Agg")  # headless execution; Colab keeps its inline backend

sns.set_theme(context="notebook", style="whitegrid")
plt.rcParams["figure.dpi"] = 110


## Load scored generations (gpt2 first)

If the scored table does not exist yet in this environment, the guard cell produces it for gpt2 through the idempotent generation CLI (the TriviaQA subset is deterministic under seed 42). Pythia scoring is heavier and typically runs through `make generate`.

In [4]:
import subprocess
from pathlib import Path

def _ensure(artifact: str, command: list[str]) -> None:
    if not Path(artifact).exists():
        print(f"producing {artifact} via:", " ".join(command))
        subprocess.run([sys.executable, *command], check=True)

_ensure("data/processed/folds.json",
        ["scripts/prepare_data.py", "--config", "config.yaml"])


In [5]:
_ensure("logs/scored_generations_gpt2.jsonl",
        ["scripts/generate_and_score.py", "--config", "config.yaml",
         "--model", "gpt2"])

In [6]:
from cassandra_ns.config import load_config
from cassandra_ns.generation.scoring import load_scored, abstention_curve

CFG = load_config("config.yaml")
MODEL = "gpt2"
scored = load_scored(CFG, MODEL)
len(scored), scored.columns.tolist()

(500,
 ['prompt_id',
  'question',
  'generation',
  'first_sentence',
  'match_status',
  'flagged',
  'correct',
  'score'])

## String-match audit

The ground truth of this phase is a normalized string-match against the aliases of each question; a defective match poisons everything downstream. This audit cell prints the match-rate summary and 50 sampled scoring decisions (fixed seed) for manual inspection; generations without a clear match are excluded from metrics and counted — never forced into correct or incorrect.

In [7]:
audit = scored.sample(min(50, len(scored)), random_state=SEED)
rate = scored["correct"].value_counts(dropna=False)
rate.to_frame("generations")

,generations
correct,
False,464
<NA>,30
True,6


In [8]:
audit[["question", "generation", "first_sentence", "match_status",
       "correct"]].reset_index(drop=True)

,question,generation,first_sentence,match_status,correct
0,Which cat spends most of his time chasing Tweety Pie?,Tweety Pie is the most popular cat in the house. He is the most popular cat in the house.\nQuestion: Which cat is t...,Tweety Pie is the most popular cat in the house.,no_alias,False
1,Which legend of the British pop world died on March the 8th 2016?,The legend of the British pop world died on March the 8th 2016.\nThe legend of the British pop world died on March ...,The legend of the British pop world died on March the 8th 2016.,no_alias,False
2,Who had a top ten hit in 1974 with Streets Of London?,The Beatles.\nSource: The Beatles' Official Website\nSource: The Beatles' Official Website\nSource: The Beatles' Of...,The Beatles.,no_alias,False
3,In the Harry Potter series of books what make and model is the Weasley family’s flying car?,The Weasley family is a family of wizards and witches. The Weasley family is a family of wizards and witches. The W...,The Weasley family is a family of wizards and witches.,no_alias,False
4,What type of animal is a macaque?,"A macaque is a small, white, male macaque that is about the size of a small dog. It is a very large animal. It is very","A macaque is a small, white, male macaque that is about the size of a small dog.",no_alias,False
5,Which film maker's first film was Pather Panchali?,The first film Pather Panchali was made by the same studio as the first film.\nThe second film Pather Panchali was ...,The first film Pather Panchali was made by the same studio as the first film.,no_alias,False
6,Which Jessica was the then oldest Oscar winner for Driving Miss Daisy?,Jessica was the youngest Oscar winner for Driving Miss Daisy.\nJessica was the first Oscar winner for Driving Miss ...,Jessica was the youngest Oscar winner for Driving Miss Daisy.,no_alias,False
7,What is the first bird mentioned in the Bible?,The first bird mentioned in the Bible is the first bird mentioned in the Bible.\nThe first bird mentioned in the Bi...,The first bird mentioned in the Bible is the first bird mentioned in the Bible.,no_alias,False
8,Who are the Brave Blossoms?,The Brave Blossoms are a group of people who are trying to save the world from the evil of the Dark Gods. They are ...,The Brave Blossoms are a group of people who are trying to save the world from the evil of the Dark Gods.,alias_in_question,<NA>
9,How many american states begin with the letter W ?,The first two states are the United States of America and the United Kingdom of Great Britain and Northern Ireland....,The first two states are the United States of America and the United Kingdom of Great Britain and Northern Ireland.,no_alias,False


## Probe score vs factual agreement

Each point is one generation: probe score (best configuration of notebook 03) against whether the generation's first declarative sentence matches the true answer, with vertical jitter to expose density where the binary outcome overlaps.

In [9]:
plot_df = scored.dropna(subset=["score", "correct"]).copy()
jitter = rng.uniform(-0.18, 0.18, size=len(plot_df))

fig, ax = plt.subplots(figsize=(8.4, 4.2))
ax.scatter(plot_df["score"], plot_df["correct"].astype(int) + jitter,
           s=14, alpha=0.5, c="#4c78a8", linewidths=0)
ax.set_yticks([0, 1], labels=["incorrect", "correct"])
ax.set_xlabel("probe score")
ax.set_ylabel("factual agreement")
fig.tight_layout()
save_figure(fig, "04_score_vs_correct")
plt.show()

## Selective abstention: coverage-accuracy curve

Sweeping the threshold tau, the model answers only generations whose probe score exceeds it. Coverage is the fraction answered; accuracy is computed on answered generations only. tau* is the threshold that maximizes accuracy subject to coverage >= min_coverage (0.6). The gain vs always answering is the operational value of the probe.

In [10]:
curve, tau_star = abstention_curve(
    plot_df["correct"].astype(int).values,
    plot_df["score"].values,
    CFG["abstention"]["min_coverage"],
)
curve.tail()

,tau,coverage,accuracy,n_answered
458,0.025176,0.991489,0.012876,466
459,0.019552,0.993617,0.012848,467
460,0.018231,0.995745,0.012821,468
461,0.007360,0.997872,0.012793,469
462,0.004942,1.000000,0.012766,470


In [11]:
star = curve[curve["coverage"] >= CFG["abstention"]["min_coverage"]]
star_row = star.loc[star["accuracy"].idxmax()]

baseline_acc = plot_df["correct"].mean()

fig, ax = plt.subplots(figsize=(7.6, 4.4))
ax.plot(curve["coverage"], curve["accuracy"], color="#4c78a8")
ax.scatter([star_row["coverage"]], [star_row["accuracy"]],
           color="#b0553a", zorder=3)
ax.annotate(f"tau* = {tau_star:.3f}\n(coverage {star_row['coverage']:.2f}, "
            f"accuracy {star_row['accuracy']:.3f})",
            (star_row["coverage"], star_row["accuracy"]),
            xytext=(8, -24), textcoords="offset points", fontsize=9)
ax.axhline(baseline_acc, ls="--", c="grey", lw=1)
ax.set_xlabel("coverage")
ax.set_ylabel("accuracy on answered")
ax.set_title(f"always-answer accuracy {baseline_acc:.3f}")
fig.tight_layout()
save_figure(fig, "04_abstention_curve")
plt.show()

star_row.to_frame("value at tau*")

,value at tau*
tau,0.644785
coverage,0.600000
accuracy,0.017730
n_answered,282.000000


## Confusion at tau*

Rows are the policy decision at tau* (answer when score >= tau*), columns the factual outcome. The off-diagonal cells are the operational risks: answered-but-wrong (residual confabulation exposure) and silent-but-right (coverage given up).

In [12]:
from sklearn.metrics import confusion_matrix

policy = (plot_df["score"] >= tau_star)
cm = confusion_matrix(plot_df["correct"].astype(int), policy,
                      labels=[True, False])

fig, ax = plt.subplots(figsize=(5.2, 4.4))
sns.heatmap(cm, annot=True, fmt="d", cmap="mako", ax=ax,
            xticklabels=["answer", "abstain"],
            yticklabels=["correct", "incorrect"])
ax.set_xlabel("policy at tau*")
ax.set_ylabel("factual outcome")
fig.tight_layout()
save_figure(fig, "04_confusion_at_tau_star")
plt.show()

## Captured confabulations (high score, incorrect)

Top-5 by probe score among incorrect generations, with the full generation text. These are the cases the abstention policy exists for: the model sounded internally confident and was factually wrong.

In [13]:
confab = (plot_df[~plot_df["correct"]]
            .nlargest(5, "score"))
for _, r in confab.iterrows():
    print(f"--- score {r['score']:.3f} | {r['question']}")
    print(r["generation"])
    print()

--- score 1.000 | In which theater did The Wizard of Oz have its New York debut?
 In the theater of the theater of the theater of the theater of the theater of the theater of the theater of the theater of the theater of the theater of the

--- score 1.000 | Honiara is the capital city of which nation?
 The capital city of the Republic of the Republic of the Republic of the Republic of the Republic of the Republic of the Republic of the Republic of the Republic of the

--- score 1.000 | Which Greek goddess is the muse of erotic poetry?
 The goddess of erotic poetry is the goddess of the goddess of the goddess of the goddess of the goddess of the goddess of the goddess of the goddess of the goddess

--- score 1.000 | Addresses in and around which British town or city have post codes beginning CA?
 The post code for the post code for the post code for the post code for the post code for the post code for the post code for the post code for

--- score 1.000 | Which element does Zn represen

## Atypical doubt (low score, correct)

Top-5 correct generations with the lowest probe scores: the model was right while its internal read looked unconfident. The gap between these two error families bounds what score-based abstention can and cannot fix.

In [14]:
doubt = (plot_df[plot_df["correct"]]
           .nsmallest(5, "score"))
for _, r in doubt.iterrows():
    print(f"--- score {r['score']:.3f} | {r['question']}")
    print(r["generation"])
    print()

--- score 0.502 | Who won baseball's first World Series of the 50s?
 The Yankees.
The Yankees won the World Series of the 50s in the 1950s.
The Yankees won the World Series of the 50s in the

--- score 0.784 | The Internet company Google is based in which US state?
 The US state of California.
Question: What is the name of the company?
Answer: Google.
Question: What is the name of the company

--- score 0.814 | The 1988 film ‘Frantic’, starring Harrison Ford, is set in which European city?
 The 1988 film ‘Frantic’, starring Harrison Ford, is set in which European city?
The film is set in the fictional city of Paris

--- score 0.934 | Anime is film animation originating in which country?
 Japan.
Source: http://www.youtube.com/watch?v=XQX-X-X-X-A
Source: http

--- score 0.998 | Pattaya, Phucket and Koh Samui are tourist destinations in which country?
 Thailand.
Source: http://www.thedailybeast.com/news/local/thedaily-beast-travel-tour



## Ensemble with the symbolic signal (optional)

When `config.ensemble.prometheus_graph` points to a PROMETHEUS-NS consistency graph, triplets extracted from the generations produce a symbolic signal and the two signals are combined with a weight learned inside validation folds. The comparison on the same extractable subset decides: if the ensemble does not improve AUC, that is the reported result.

In [15]:
if CFG["ensemble"]["enabled"] and CFG["ensemble"]["prometheus_graph"]:
    from cassandra_ns.ensemble.symbolic_bridge import load_ensemble_results
    ens = load_ensemble_results(CFG)
    ens
else:
    print("ensemble unavailable: "
          "prometheus_graph not configured in config.yaml")

## Interpretation and limits

- **What the (coverage, accuracy) gain at tau* would buy in a production abstention policy — and that it is measured on this test set only.**
- **That probe scores are correlational:** a high score is evidence the residual stream looks truthful-like, not proof of a fact-checker.
- **That generations come from greedy decoding of small models:** the confabulation rate here does not transfer to frontier models.
- **That the string-match ground truth is statistical, not semantic:** equivalent answer phrasings outside the aliases count as incorrect.

This notebook is the qualitative face of the quantitative claims in notebook 03.